# Hugging Face audio tasks — task-specific models
**6 tasks · 6 task-specific models · 2 examples per task**

Python 3.11.9 recommended. Models download from Hugging Face and run **locally**, using a token loaded from `.env`. There are no paid hosted inference calls in this notebook. Hardware, electricity, internet and any rented GPU can cost money. Model licenses still apply; free download does not mean unrestricted use.

These notebooks use task-specific Transformers/Diffusers models directly. Each example calls the model pipeline as a regular Python function, without LangChain wrappers.

Run setup, then each task's loading cell and both example cells. Cleanup releases RAM. All tasks can run on CPU. The TTS task additionally uses a dedicated vocoder. Short audio clips keep inference practical. Six tasks are covered; audio question answering, music generation and source separation are not included.

Validation: JSON, cell syntax and example counts checked. Full model downloads and inference were not executed during creation.

## Install
Use a clean environment. Run once and restart the kernel. Transformers v4 is intentional. This is a constrained environment, not a fully tested lockfile.

In [ ]:
%pip install -q "torch>=2.6,<3" "torchvision>=0.21,<1" "transformers==4.57.6" "langchain-huggingface==0.3.1" "langchain-core>=0.3.70,<1" "huggingface-hub>=0.34,<1" "python-dotenv>=1,<2" "numpy>=1.26,<3" "librosa>=0.10,<0.12" "soundfile>=0.12,<1" "scipy>=1.11,<2" "sentencepiece>=0.2,<0.3" "protobuf>=4,<7"

## Load token from `.env`
Create `.env` beside the notebook (or upload it to `/content` in Colab):
```dotenv
HUGGINGFACE_API_KEY=hf_replace_with_your_token
```
Use a Hugging Face read token from https://huggingface.co/settings/tokens. It authenticates model downloads; it is not an OpenAI API key.

In [ ]:
import os, gc
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env")
HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not HF_TOKEN or HF_TOKEN == "hf_replace_with_your_token":
    raise ValueError("Add your token to .env, then rerun this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
import torch
import numpy as np
from transformers import pipeline, set_seed
from IPython.display import display, Audio
set_seed(42)
DEVICE = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if DEVICE == 0 else "CPU")

## Audio inputs and resampling
First synthesize two speech clips in Task 1. Tasks 2 and 6 use those clips. Tasks 3–4 download two real sounds from librosa's example collection (trumpet and bird song). Task 5 needs two short non-English WAV recordings supplied by you for a meaningful translation demo. Optional Colab upload code is included there. Audio inputs are converted to mono and resampled to the actual model rate; no fake sample-rate relabeling.

In [ ]:
import librosa
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

def load_audio(path, target_sr):
    samples, source_sr = sf.read(path, dtype="float32")
    if samples.ndim == 2:
        samples = samples.mean(axis=1)
    if source_sr != target_sr:
        factor = gcd(int(source_sr), int(target_sr))
        samples = resample_poly(samples, target_sr // factor, source_sr // factor).astype("float32")
    # Keep the first 10 seconds for a short classroom example.
    samples = np.asarray(samples[:target_sr * 10], dtype="float32")
    return {"raw": samples, "sampling_rate": target_sr}


## Task 1: Text-to-speech
**Model:** [microsoft/speecht5_tts](https://huggingface.co/microsoft/speecht5_tts)

MIT-licensed model; uses microsoft/speecht5_hifigan as a vocoder. Uses a public example speaker embedding from the MIT-licensed CMU Arctic x-vector collection for consistent synthesis. Two generated clips are reused as ASR inputs.

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts", token=HF_TOKEN)
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts", token=HF_TOKEN).eval()
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan", token=HF_TOKEN).eval()
from huggingface_hub import hf_hub_download
import zipfile
speaker_archive = hf_hub_download(
    repo_id="Matthijs/cmu-arctic-xvectors", filename="spkrec-xvect.zip",
    repo_type="dataset", token=HF_TOKEN,
)
with zipfile.ZipFile(speaker_archive) as archive:
    speaker_file = next(name for name in sorted(archive.namelist()) if name.endswith(".npy"))
    with archive.open(speaker_file) as stream:
        speaker = torch.tensor(np.load(stream), dtype=torch.float32).reshape(1, 512)

def synthesize(text):
    inputs = processor(text=text, return_tensors="pt")
    with torch.inference_mode():
        audio = model.generate_speech(inputs["input_ids"], speaker, vocoder=vocoder)
    return audio.cpu().numpy()


### Example 1

In [ ]:
audio = synthesize('Welcome to the artificial intelligence training session.')
sf.write("speech_1.wav", audio, 16000)
display(Audio(audio, rate=16000))

### Example 2

In [ ]:
audio = synthesize('Please contact customer support if you need help with your account.')
sf.write("speech_2.wav", audio, 16000)
display(Audio(audio, rate=16000))

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, model, vocoder, processor, speaker
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 2: English speech-to-text
**Model:** [openai/whisper-tiny.en](https://huggingface.co/openai/whisper-tiny.en)

Transcribes the two synthetic clips from Task 1. This is English-only Whisper; a different multilingual Whisper checkpoint is used for translation later.

In [ ]:
pipe = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en", token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
clip = load_audio("speech_1.wav", pipe.feature_extractor.sampling_rate)
print(pipe(clip)["text"])

### Example 2

In [ ]:
clip = load_audio("speech_2.wav", pipe.feature_extractor.sampling_rate)
print(pipe(clip)["text"])

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Download two environmental sound samples
Internet is needed on the first run. The librosa example collection has its own data attribution/license; these are demonstration inputs, not a training corpus.

In [ ]:
sound1 = librosa.ex("trumpet")
sound2 = librosa.ex("robin")
for path in [sound1, sound2]:
    samples, rate = sf.read(path)
    display(Audio(samples.T, rate=rate))

## Task 3: Environmental sound classification
**Model:** [MIT/ast-finetuned-audioset-10-10-0.4593](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593)

Recognizes learned AudioSet sound categories. Short clips may contain multiple sounds; the top predictions are not guaranteed correct.

In [ ]:
pipe = pipeline("audio-classification", model="MIT/ast-finetuned-audioset-10-10-0.4593", token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
clip = load_audio(sound1, pipe.feature_extractor.sampling_rate)
print(pipe(clip, top_k=5))

### Example 2

In [ ]:
clip = load_audio(sound2, pipe.feature_extractor.sampling_rate)
print(pipe(clip, top_k=5))

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 4: Zero-shot audio classification
**Model:** [laion/clap-htsat-unfused](https://huggingface.co/laion/clap-htsat-unfused)

Compares audio to supplied text descriptions; it does not require retraining for each candidate label. This model requires 48 kHz audio, so load_audio resamples it.

In [ ]:
pipe = pipeline("zero-shot-audio-classification", model="laion/clap-htsat-unfused", token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
clip = load_audio(sound1, pipe.feature_extractor.sampling_rate)
labels = ["The sound of a trumpet playing music", "The sound of a bird singing", "The sound of a dog barking"]
print(pipe(clip, candidate_labels=labels))

### Example 2

In [ ]:
clip = load_audio(sound2, pipe.feature_extractor.sampling_rate)
labels = ["The sound of a trumpet playing music", "The sound of a bird singing", "The sound of a dog barking"]
print(pipe(clip, candidate_labels=labels))

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 5: Speech translation into English
**Model:** [openai/whisper-tiny](https://huggingface.co/openai/whisper-tiny)

A multilingual speech model. Provide two short non-English WAV recordings, such as Hindi or French, as translation_1.wav and translation_2.wav. Output is English text, not translated speech. Tiny Whisper can be weak on some languages or accents; review translations.

### Optional Colab upload
For local Jupyter, copy the WAV files into the working folder. In Colab, uncomment and run this cell, then rename the uploaded files if needed.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # Upload translation_1.wav and translation_2.wav

In [ ]:
pipe = pipeline("automatic-speech-recognition", model="openai/whisper-tiny", token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
path = Path("translation_1.wav")
if path.exists():
    clip = load_audio(path, pipe.feature_extractor.sampling_rate)
    print(pipe(clip, generate_kwargs={"task": "translate"})["text"])
else:
    print("Upload translation_1.wav (non-English speech), then rerun this example.")

### Example 2

In [ ]:
path = Path("translation_2.wav")
if path.exists():
    clip = load_audio(path, pipe.feature_extractor.sampling_rate)
    print(pipe(clip, generate_kwargs={"task": "translate"})["text"])
else:
    print("Upload translation_2.wav (non-English speech), then rerun this example.")

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 6: Speech emotion classification
**Model:** [superb/wav2vec2-base-superb-er](https://huggingface.co/superb/wav2vec2-base-superb-er)

Returns dataset-specific emotion labels (such as neutral, happy, angry, sad). Demo inputs reuse the TTS clips and may sound neutral; replace them with two expressive WAV recordings for a better demonstration. Emotion predictions are not reliable judgments about a person.

In [ ]:
pipe = pipeline("audio-classification", model="superb/wav2vec2-base-superb-er", token=HF_TOKEN, device=DEVICE)
print("Label mapping:", pipe.model.config.id2label)


### Example 1

In [ ]:
clip = load_audio("speech_1.wav", pipe.feature_extractor.sampling_rate)
print(pipe(clip, top_k=4))

### Example 2

In [ ]:
clip = load_audio("speech_2.wav", pipe.feature_extractor.sampling_rate)
print(pipe(clip, top_k=4))

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Cost and troubleshooting
No paid hosted inference calls are made. Downloadable weights are used locally; GPU rental, data transfer and electricity can still cost money. Each linked model card governs licensing and use. Sample audio licenses are separate.

Run Task 1 before Tasks 2 and 6. For speech translation, upload your own two non-English WAV files; the notebook prints a helpful message if absent. Sound samples download automatically. Use short WAV files to avoid FFmpeg/MP3 dependencies. If imports fail, install in a clean Python 3.11 environment and restart the kernel. If GPU memory runs out, clean up or restart and run one task at a time. Classifier scores are not measured accuracy.

[Transformers audio pipelines](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [SpeechT5 model](https://huggingface.co/microsoft/speecht5_tts)